In [9]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

from Pipeline.Global.GallstoneDataSet import GallstoneDataSet
from Pipeline.Algorithm.ExtremeLearningMachine import ExtremeLearningMachine
from Pipeline.Methodology.EvaluationMatrix import EvaluationMatrix
from Pipeline.Global.GlobalSetting import GlobalSetting

In [10]:
model_configs = GlobalSetting.get_model_configs()

In [11]:
gallstone_dataset = GallstoneDataSet()
gallstone_dataset.fetch_data_path_1()
gallstone_dataset.cross_validate_test(5)

In [12]:
testing_results = []
for config in model_configs:
    for fold_idx, (x_tr_raw, y_tr_raw, x_te_raw, y_te_raw) in enumerate(gallstone_dataset.fold_split):

        scaler = MinMaxScaler()
        x_tr_scaled = scaler.fit_transform(x_tr_raw.values)
        x_te_scaled = scaler.transform(x_te_raw.values)

        for seed in GlobalSetting.elm_initial_state_range:
            elm = ExtremeLearningMachine(
                features_size   = x_tr_scaled.shape[1],
                hidden_size     = config["Hidden_Nodes"],
                activation_function     = config["Activation"],
                regularization_lambda   = config["Lambda_Value"]
            )
            elm.initialize_random_weights(random_seed = seed)

            elm.fit(x_tr_scaled, y_tr_raw.values)
            y_pred = elm.predict(x_te_scaled)

            evaluation = EvaluationMatrix(y_te_raw.values, y_pred)

            metrics = evaluation.get_all_metrics()
            test_result = {
                "Model_Type"   : config.get('Model_Types', 'Unknown'),
                "Hidden_Nodes" : config['Hidden_Nodes'],
                "Lambda_Value" : config['Lambda_Value'],
                "Activation"   : config['Activation'].__name__,
                "Fold_ID"      : fold_idx,
                "Seed"         : seed
            }
            test_result.update(metrics)
            testing_results.append(test_result)

In [13]:
# 1. Convert to DataFrame
df_results = pd.DataFrame(testing_results)

# 2. Define your unique configuration signature
group_cols = ["Model_Type", "Hidden_Nodes", "Lambda_Value", "Activation"]

# 3. Extract metrics, explicitly ignoring BOTH Seed and Fold_ID
metric_cols = [col for col in df_results.columns if col not in group_cols + ["Seed", "Fold_ID"]]

# 4. Define Aggregations
agg_funcs = {metric: ['mean','sem' ,'std', 'max' , 'min'] for metric in metric_cols}

# 5. Execute Group By
summary_df = df_results.groupby(group_cols).agg(agg_funcs).reset_index()

# Flatten Multi-Index Columns
summary_df.columns = [
    f"{col[0]}_{col[1]}" if col[1] else col[0]
    for col in summary_df.columns.values
]

summary_df

,Model_Type,Hidden_Nodes,Lambda_Value,Activation,Accuracy_mean,Accuracy_sem,Accuracy_std,Accuracy_max,Accuracy_min,Precision_mean,...,Bal Accuracy_mean,Bal Accuracy_sem,Bal Accuracy_std,Bal Accuracy_max,Bal Accuracy_min,MCC_mean,MCC_sem,MCC_std,MCC_max,MCC_min
0,Best_Hidden_Nodes,48,0.000000,sigmoid,0.752194,0.004521,0.055374,0.857143,0.609375,0.787264,...,0.751560,0.004519,0.055345,0.856855,0.609375,0.511332,0.008914,0.109172,0.717052,0.221470
1,Best_Lambda,48,0.031250,sigmoid,0.763839,0.004415,0.054078,0.888889,0.640625,0.808238,...,0.763075,0.004411,0.054028,0.889113,0.640625,0.536489,0.008720,0.106801,0.778226,0.288231
2,Best_Size_and_Lambda,48,0.031250,sigmoid,0.763839,0.004415,0.054078,0.888889,0.640625,0.808238,...,0.763075,0.004411,0.054028,0.889113,0.640625,0.536489,0.008720,0.106801,0.778226,0.288231
3,Grid_Combination,93,0.062500,sigmoid,0.766782,0.004633,0.056745,0.888889,0.656250,0.811568,...,0.765983,0.004629,0.056691,0.888609,0.656250,0.541806,0.009080,0.111212,0.780948,0.322749
4,Grid_Optimization,37,0.077887,sigmoid,0.756438,0.004376,0.053600,0.904762,0.656250,0.797755,...,0.755719,0.004381,0.053661,0.904234,0.656250,0.520452,0.008680,0.106308,0.810924,0.314970


In [14]:
# 1. Convert to DataFrame
df_results2 = pd.DataFrame(testing_results)

# 2. Define the signature: Group by Config AND Fold_ID
# This ensures we calculate the variance of the SEEDS strictly within each fold
group_cols = ["Model_Type", "Hidden_Nodes", "Lambda_Value", "Activation", "Fold_ID"]

# 3. Extract metrics, explicitly ignoring ONLY "Seed"
metric_cols = [col for col in df_results2.columns if col not in group_cols + ["Seed"]]

# 4. Define Aggregations (mean and std across the random seeds for THIS specific fold)
agg_funcs = {metric: ['mean', 'std', 'max', 'min'] for metric in metric_cols}

# 5. Execute Group By
summary_df_by_fold = df_results2.groupby(group_cols).agg(agg_funcs).reset_index()

# 6. Flatten Multi-Index Columns
summary_df_by_fold.columns = [
    f"{col[0]}_{col[1]}" if col[1] else col[0]
    for col in summary_df_by_fold.columns.values
]

# Sort by Model Type, then Fold ID to make it highly readable
summary_df_by_fold = summary_df_by_fold.sort_values(by=["Model_Type", "Fold_ID"]).reset_index(drop=True)

summary_df_by_fold

,Model_Type,Hidden_Nodes,Lambda_Value,Activation,Fold_ID,Accuracy_mean,Accuracy_std,Accuracy_max,Accuracy_min,Precision_mean,...,F2-Score_max,F2-Score_min,Bal Accuracy_mean,Bal Accuracy_std,Bal Accuracy_max,Bal Accuracy_min,MCC_mean,MCC_std,MCC_max,MCC_min
0,Best_Hidden_Nodes,48,0.000000,sigmoid,0,0.746875,0.028840,0.781250,0.656250,0.782333,...,0.791139,0.570470,0.744412,0.029204,0.781036,0.652981,0.498254,0.058353,0.577284,0.313404
1,Best_Hidden_Nodes,48,0.000000,sigmoid,1,0.725000,0.041765,0.812500,0.656250,0.811906,...,0.714286,0.486111,0.725000,0.041765,0.812500,0.656250,0.469327,0.085658,0.657952,0.313112
2,Best_Hidden_Nodes,48,0.000000,sigmoid,2,0.795833,0.030135,0.843750,0.734375,0.787732,...,0.903614,0.732484,0.795833,0.030135,0.843750,0.734375,0.593601,0.061036,0.699913,0.470824
3,Best_Hidden_Nodes,48,0.000000,sigmoid,3,0.691146,0.037537,0.750000,0.609375,0.725673,...,0.750000,0.522876,0.691146,0.037537,0.750000,0.609375,0.387796,0.075390,0.503953,0.221470
4,Best_Hidden_Nodes,48,0.000000,sigmoid,4,0.802116,0.040786,0.857143,0.714286,0.828675,...,0.844156,0.612245,0.801411,0.040827,0.856855,0.712198,0.607684,0.081790,0.717052,0.440689
5,Best_Lambda,48,0.031250,sigmoid,0,0.755729,0.033205,0.796875,0.671875,0.808993,...,0.779221,0.582192,0.752672,0.033276,0.796188,0.669110,0.519596,0.068863,0.628155,0.344160
6,Best_Lambda,48,0.031250,sigmoid,1,0.747917,0.030907,0.796875,0.671875,0.854924,...,0.741935,0.540541,0.747917,0.030907,0.796875,0.671875,0.520568,0.064396,0.632280,0.345271
7,Best_Lambda,48,0.031250,sigmoid,2,0.792708,0.031232,0.843750,0.734375,0.784946,...,0.878788,0.723270,0.792708,0.031232,0.843750,0.734375,0.587131,0.062558,0.688847,0.468979
8,Best_Lambda,48,0.031250,sigmoid,3,0.694271,0.024853,0.734375,0.640625,0.731907,...,0.700637,0.533333,0.694271,0.024853,0.734375,0.640625,0.394308,0.048353,0.474579,0.288231
9,Best_Lambda,48,0.031250,sigmoid,4,0.828571,0.029000,0.888889,0.761905,0.860417,...,0.897436,0.675676,0.827806,0.029189,0.889113,0.760081,0.660841,0.057843,0.778226,0.535496


In [15]:
GlobalSetting.save_dataframe_to_record(summary_df,"Model_Testing_Result.csv")

[I/O Trace] Record exported successfully: ../../Storage/Record\Model_Testing_Result.csv
